# 🛡️ AEGIS fine-tune — scam / injection classifier

Fine-tunes **MobileBERT** (25M params) as a 3-class classifier on the AEGIS training seeds:
`0 = legit`, `1 = scam`, `2 = injection`.

**How to run (5 minutes):**
1. Sign in to Colab with your Google account.
2. `Runtime` → `Change runtime type` → **T4 GPU** → Save.
3. `Runtime` → `Run all`.
4. At the end, a file `aegis-finetuned-model.zip` downloads automatically — hand it back (it never leaves your session except this download).

**Privacy:** the training data is a public file from the AEGIS GitHub repo. Nothing about you is sent anywhere; the model weights produced here are trained only on the public corpus.

In [ ]:
!pip install -q onnx onnxruntime onnxscript

In [ ]:
import json, urllib.request

URL = "https://raw.githubusercontent.com/starboy-hub/AEGIS/main/evaluation/training/train-seeds.json"
try:
    urllib.request.urlretrieve(URL, "train-seeds.json")
    data = json.load(open("train-seeds.json"))
except Exception as e:
    print("Fetch failed (" + str(e) + ") — upload evaluation/training/train-seeds.json manually:")
    from google.colab import files
    up = files.upload()
    data = json.load(open(list(up.keys())[0]))

msgs = data["messages"]
from collections import Counter
print("seeds:", len(msgs), Counter(m["label"] for m in msgs))
print("stats:", data.get("stats"))

In [ ]:
# 3-class label map (0 legit / 1 scam / 2 injection) + light oversampling of
# the rare injection class so it isn't drowned out
LABELS = {"legit": 0, "scam": 1, "injection": 2}
texts, labels = [], []
for m in msgs:
    n = 3 if m["label"] == "injection" else 1
    for _ in range(n):
        texts.append(m["text"])
        labels.append(LABELS[m["label"]])

from sklearn.model_selection import train_test_split
from datasets import Dataset
tr_x, va_x, tr_y, va_y = train_test_split(texts, labels, test_size=0.15, stratify=labels, random_state=42)
train_ds = Dataset.from_dict({"text": tr_x, "label": tr_y})
val_ds = Dataset.from_dict({"text": va_x, "label": va_y})
print("train:", len(train_ds), " val:", len(val_ds))


In [ ]:
from transformers import AutoTokenizer
MODEL = "google/mobilebert-uncased"
tok = AutoTokenizer.from_pretrained(MODEL)
def tok_fn(batch):
    return tok(batch["text"], truncation=True, max_length=128)
train_tok = train_ds.map(tok_fn, batched=True)
val_tok = val_ds.map(tok_fn, batched=True)

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)
def metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

args = TrainingArguments(
    output_dir="aegis_ft",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to=[]
)
trainer = Trainer(model=model, args=args,
                  train_dataset=train_tok, eval_dataset=val_tok,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()


In [ ]:
# Per-class report — the numbers that matter
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(val_tok)
preds = np.argmax(pred.predictions, axis=-1)
names = ["legit", "scam", "injection"]
print(classification_report(pred.label_ids, preds, target_names=names, digits=3))
print("confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(pred.label_ids, preds))
report = classification_report(pred.label_ids, preds, target_names=names, digits=3)

In [ ]:
# Export ONNX + quantize + package for AEGIS integration (no optimum — Colab-safe)
import torch, json, shutil, os
model.cpu(); model.eval()
sample = tok("AEGIS export check", return_tensors="pt")
os.makedirs("ft_onnx", exist_ok=True)
torch.onnx.export(
    model,
    (sample["input_ids"], sample["token_type_ids"], sample["attention_mask"]),
    "ft_onnx/model.onnx",
    input_names=["input_ids", "token_type_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch", 1: "sequence"},
                  "token_type_ids": {0: "batch", 1: "sequence"},
                  "attention_mask": {0: "batch", 1: "sequence"},
                  "logits": {0: "batch"}},
    opset_version=14
)
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic("ft_onnx/model.onnx", "ft_onnx/model_quantized.onnx", weight_type=QuantType.QInt8)

os.makedirs("pkg", exist_ok=True)
tok.save_pretrained("pkg")
shutil.copy("ft_onnx/model.onnx", "pkg/model.onnx")
shutil.copy("ft_onnx/model_quantized.onnx", "pkg/model_quantized.onnx")
cfg = os.path.join(trainer.state.best_model_checkpoint, "config.json")
if os.path.exists(cfg): shutil.copy(cfg, "pkg/config.json")
open("pkg/README.txt", "w").write("AEGIS fine-tuned classifier\nclasses: 0=legit, 1=scam, 2=injection\n\nvalidation report:\n" + report)
shutil.make_archive("aegis-finetuned-model", "zip", "pkg")
from google.colab import files
files.download("aegis-finetuned-model.zip")
print("done — aegis-finetuned-model.zip downloaded")
